# On-Disk Inductive Learning: Large-Scale Datasets with TopoBench

This tutorial demonstrates TopoBench's **on-disk preprocessing** for training on large inductive datasets (many graphs) that exceed available RAM.

**Key Features Covered:**
- ✅ Constant memory preprocessing (O(1) per sample)
- ✅ Topological transform support (SimplicialCliqueLifting, etc.)
- ✅ Transform caching for reuse
- ✅ Unified factory interface
- ✅ Seamless TopoBench integration

## Why On-Disk?

Traditional in-memory preprocessing loads ALL topological structures into RAM:

**Problem:**
- Large datasets → millions of structures → **RAM exhaustion (OOM)**
- Example: 5000 graphs × 80 nodes × 200 triangles each = **~6GB just for structures!**

**Solution:**
- Process graphs **one-by-one**, stream to disk → **constant memory (~50-100MB)**
- Supports **all TopoBench transforms** (liftings)
- Training loads from disk as needed

**Use on-disk when:**
- Dataset has many graphs (>1000)
- Graphs are large (>50 nodes) or high degree
- Using topological liftings (triangles, cliques)
- Limited RAM (<8GB available)

## Prerequisites

```bash
pip install torch torch-geometric networkx omegaconf pytorch-lightning
```

## Step 1: Create Your Dataset Class

Follow standard TopoBench pattern - inherit from `InMemoryDataset`:

In [ ]:
import networkx as nx
import torch
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.io import fs
from omegaconf import DictConfig

class MyLargeInductiveDataset(InMemoryDataset):
    """Custom large inductive dataset.
    
    This creates a source dataset. On-disk preprocessing
    will handle the topological structures efficiently.
    """
    
    def __init__(self, root, name, parameters: DictConfig):
        self.name = name
        self.parameters = parameters
        super().__init__(root)
        
        # Load processed data
        out = fs.torch_load(self.processed_paths[0])
        if len(out) == 4:
            data, self.slices, self.sizes, data_cls = out
            self.data = data_cls.from_dict(data) if isinstance(data, dict) else data
        else:
            data, self.slices, self.sizes = out
            self.data = data
    
    @property
    def raw_file_names(self):
        return []
    
    @property
    def processed_file_names(self):
        return "data.pt"
    
    def download(self):
        pass  # Implement if downloading from external source
    
    def process(self):
        """Generate your graphs here."""
        data_list = []
        
        # Example: Generate synthetic graphs (replace with your data)
        for i in range(self.parameters.num_graphs):
            G = nx.watts_strogatz_graph(
                n=self.parameters.nodes_per_graph,
                k=self.parameters.degree,
                p=0.3,
                seed=42+i
            )
            
            # Convert to PyG Data
            edges = list(G.edges())
            edge_index = torch.tensor(edges, dtype=torch.long).t()
            edge_index = torch.cat([edge_index, edge_index[[1, 0]]], dim=1)  # Undirected
            
            x = torch.randn(G.number_of_nodes(), self.parameters.num_features)
            y = torch.randint(0, self.parameters.num_classes, (1,))
            
            data = Data(x=x, edge_index=edge_index, y=y, num_nodes=G.number_of_nodes())
            data_list.append(data)
        
        # Collate and save
        self.data, self.slices = self.collate(data_list)
        fs.torch_save(
            (self._data.to_dict(), self.slices, {}, self._data.__class__),
            self.processed_paths[0]
        )

## Step 2: Create Your Loader

Inherit from `AbstractLoader` following TopoBench conventions:

In [ ]:
from topobench.data.loaders.base import AbstractLoader

class MyLargeInductiveLoader(AbstractLoader):
    """Loader for custom inductive dataset."""
    
    def __init__(self, parameters: DictConfig):
        super().__init__(parameters)
    
    def load_dataset(self):
        dataset = MyLargeInductiveDataset(
            root=str(self.root_data_dir),
            name=self.parameters.data_name,
            parameters=self.parameters
        )
        return dataset

## Step 3: Use On-Disk Preprocessing with Transforms 🚀

**Key Feature:** Full support for TopoBench topological transforms (liftings)!

In [ ]:
from omegaconf import OmegaConf
from topobench.data.preprocessor import OnDiskInductivePreprocessor

# Configure your dataset
loader_config = OmegaConf.create({
    "data_dir": "./data/",
    "data_name": "MyLargeDataset",
    "num_graphs": 5000,
    "nodes_per_graph": 80,
    "degree": 15,
    "num_features": 16,
    "num_classes": 5
})

# Load source dataset
loader = MyLargeInductiveLoader(loader_config)
dataset, dataset_dir = loader.load()
print(f"Loaded {len(dataset)} graphs")

# ✨ Configure topological transforms (liftings)
transforms_config = OmegaConf.create({
    "clique_lifting": {
        "transform_type": "lifting",
        "transform_name": "SimplicialCliqueLifting",
        "complex_dim": 2  # Include up to triangles
    }
})

# Create on-disk preprocessor with transforms
ondisk_dataset = OnDiskInductivePreprocessor(
    dataset=dataset,
    data_dir="./processed/large_dataset",
    transforms_config=transforms_config,  # ← Transforms applied during preprocessing!
    force_reload=False  # Reuses cached data if config unchanged
)

print(f"✓ On-disk preprocessing complete")
print(f"  - Samples: {len(ondisk_dataset)}")
print(f"  - Memory: Constant (~50-100MB)")
print(f"  - Transforms: Cached on disk for reuse")

### 🎯 Alternative: Use Factory Function (Simpler!)

The `create_preprocessor()` factory provides a unified interface that auto-detects when to use on-disk:

In [ ]:
from topobench.data.preprocessor import create_preprocessor

# Unified interface - same for in-memory or on-disk!
preprocessor = create_preprocessor(
    dataset=dataset,
    data_dir="./processed/auto",
    transforms_config=transforms_config,
    mode="auto"  # Options: "auto", "inmemory", "ondisk"
    # Auto mode: uses on-disk if dataset is large or RAM limited
)

print(f"✓ Preprocessor created: {type(preprocessor).__name__}")
# Will be OnDiskInductivePreprocessor for large datasets

## Step 4: Load Dataset Splits

On-disk datasets support the same split loading as standard TopoBench:

In [ ]:
from topobench.data.utils import load_inductive_splits

# Configure splits
split_config = OmegaConf.create({
    "learning_setting": "inductive",
    "split_type": "random",
    "train_prop": 0.5,
    "val_prop": 0.25,
})

# Load splits (built-in support)
train, val, test = ondisk_dataset.load_dataset_splits(split_config)

print(f"Splits created:")
print(f"  - Train: {len(train)} samples")
print(f"  - Val: {len(val)} samples")
print(f"  - Test: {len(test)} samples")

## Step 5: Create Dataloader

Use standard TopoBench `TBDataloader`:

In [ ]:
from topobench.dataloader import TBDataloader

# Create dataloader (works identically to in-memory)
datamodule = TBDataloader(
    dataset_train=train,
    dataset_val=val,
    dataset_test=test,
    batch_size=32,
    num_workers=0  # Set >0 for multi-process loading
)

print("✓ Dataloader ready")

## Step 6: Train Your Model

Training proceeds normally using TopoBench models:

In [ ]:
from lightning import Trainer
from topobench.model import TBModel
from topobench.nn.backbones.simplicial import SCCNNCustom
from topobench.nn.readouts.simplicial_readout import SimplicialReadout
from topobench.loss import TBLoss
from topobench.optimizer import TBOptimizer

# Model configuration
HIDDEN_DIM = 64
OUT_CHANNELS = 5

# Create model components
backbone = SCCNNCustom(
    in_channels_all=(16, HIDDEN_DIM, HIDDEN_DIM),
    hidden_channels_all=(HIDDEN_DIM, HIDDEN_DIM, HIDDEN_DIM),
    conv_order=1,
    sc_order=2,
    n_layers=2
)

readout = SimplicialReadout(
    in_channels=HIDDEN_DIM,
    out_channels=OUT_CHANNELS,
    task_level="graph"
)

loss = TBLoss(dataset_loss={
    "task": "classification",
    "loss_type": "cross_entropy"
})

optimizer = TBOptimizer(
    optimizer_id="Adam",
    parameters={"lr": 0.01}
)

# Create TopoBench model
model = TBModel(
    backbone=backbone,
    readout=readout,
    loss=loss,
    optimizer=optimizer
)

# Train with Lightning
trainer = Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    enable_progress_bar=True
)

trainer.fit(model, datamodule)

print("✅ Training complete!")
print("   Memory stayed constant throughout training.")

## 🔍 Key Features in Action

### Transform Caching

Transforms are cached by parameter hash - reuse across runs:

In [ ]:
# First run: processes all graphs
ondisk_v1 = OnDiskInductivePreprocessor(
    dataset=dataset,
    data_dir="./processed/cached",
    transforms_config=transforms_config
)
# Processing: ~30 seconds for 5000 graphs

# Second run with SAME config: instant load from cache
ondisk_v2 = OnDiskInductivePreprocessor(
    dataset=dataset,
    data_dir="./processed/cached",
    transforms_config=transforms_config  # Same config
)
# Loading: <1 second! ⚡

print("✓ Transform cache reused - instant loading!")

### Supported Transforms

All TopoBench transforms work with on-disk preprocessing:

In [ ]:
# Example: Different transforms

# 1. Simplicial Clique Lifting (triangles, tetrahedra)
config_clique = OmegaConf.create({
    "clique_lifting": {
        "transform_type": "lifting",
        "transform_name": "SimplicialCliqueLifting",
        "complex_dim": 3  # Up to 4-cliques
    }
})

# 2. Hypergraph K-Hop Lifting
config_khop = OmegaConf.create({
    "khop_lifting": {
        "transform_type": "lifting",
        "transform_name": "HypergraphKHopLifting",
        "k_value": 2,
        "signed": False
    }
})

# Use any transform - on-disk preprocessing handles it!
print("✓ All TopoBench transforms supported")

## 📊 Performance Comparison

| Aspect | In-Memory | On-Disk |
|--------|-----------|----------|
| **Memory** | O(N × D²) structures | O(1) constant |
| **5K graphs example** | ~6GB RAM | ~80MB RAM |
| **Preprocessing** | All in RAM at once | Stream to disk |
| **Training speed** | Baseline | ~1.2× slower (disk I/O) |
| **Transform support** | ✅ Yes | ✅ Yes |
| **Caching** | None | ✅ Persistent |
| **Max dataset size** | Limited by RAM | Limited by disk |

**When to use on-disk:**
- ✅ Dataset > 1000 graphs
- ✅ Graphs > 50 nodes or high degree
- ✅ Using topological liftings
- ✅ RAM < 8GB or shared system

## 💡 Tips & Best Practices

1. **Start with small subset** to test your pipeline
2. **Use `force_reload=True`** if you change transform parameters
3. **Monitor disk space** - processed samples take ~2-5× original size
4. **Use SSD** for faster I/O during training
5. **Cache directory** is reusable across experiments with same config

## 📚 Summary

**What you learned:**
1. ✅ Create on-disk datasets with `OnDiskInductivePreprocessor`
2. ✅ Apply topological transforms (liftings) during preprocessing
3. ✅ Use transform caching for efficiency
4. ✅ Use factory function `create_preprocessor()` for simpler code
5. ✅ Train normally with TopoBench models and Lightning

**Key advantage:** Train on datasets that would OOM with in-memory approach! 🎊

**Next steps:**
- Try with your own datasets
- Experiment with different transforms
- Check out `tutorial_ondisk_transductive.ipynb` for large graph learning
- See `OGBN_PRODUCTS_GUIDE.md` for real-world example (2.4M nodes)